# 01 · Data preparation
Converts the thesis v2.0 parquet splits to the §5.1 schema, verifies them against the ID files
and the §5.1 counts, checks split hygiene, and writes the frozen eval-set ID lists.

**Run once, top to bottom.** Commit `data/eval_sets/*.txt`; paste the final log entry into `RESEARCH_LOG.md`.

Outputs: `data/processed/{train,val,test}.jsonl` · `data/eval_sets/{test_full,test_optB,pilot_val_300}.txt`

In [1]:
import json, sys, platform, unicodedata, datetime
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR  = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
EVAL_DIR = ROOT / "data" / "eval_sets"
for d in (PROC_DIR, EVAL_DIR):
    d.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50, "display.width", 200, "display.max_colwidth", 80)
print("ROOT   :", ROOT)
print("Python :", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)
print("Raw files:", sorted(p.name for p in RAW_DIR.iterdir()))

ROOT   : e:\sinhala-sarcasm-llm
Python : 3.13.15 | pandas 3.1.1 | numpy 2.5.5
Raw files: ['test_ids.csv', 'test_v2.0.parquet', 'train_ids.csv', 'train_v2.0.parquet', 'val_ids.csv', 'val_v2.0.parquet']


In [2]:
SPLITS = ("train", "val", "test")
RAW_FILES = {s: RAW_DIR / f"{s}_v2.0.parquet" for s in SPLITS}
ID_FILES  = {s: RAW_DIR / f"{s}_ids.csv"      for s in SPLITS}

def read_parquet_as_text(path: Path) -> pd.DataFrame:
    df = pd.read_parquet(path)
    # everything as text (ids keep leading zeros); missing -> ""
    return df.astype(str).where(df.notna(), "")

raw     = {s: read_parquet_as_text(RAW_FILES[s]) for s in SPLITS}
ids_raw = {s: pd.read_csv(ID_FILES[s], dtype=str, keep_default_na=False) for s in SPLITS}

for s in SPLITS:
    print(f"{s:5s} parquet {raw[s].shape}   id-file {ids_raw[s].shape}")
print("\ncolumns:", list(raw["train"].columns))

train parquet (32582, 22)   id-file (32582, 1)
val   parquet (6938, 22)   id-file (6938, 1)
test  parquet (6875, 22)   id-file (6875, 1)

columns: ['id', 'post_id', 'post_group', 'source_file', 'source_url', 'text', 'context', 'context_input', 'sentiment_label', 'sarcasm_label', 'script_type', 'has_context', 'has_emoji', 'n_emoji', 'n_words', 'n_chars', 'had_url', 'sentiment_id', 'sarcasm_id', 'text_a', 'text_b', 'split']


In [3]:
allrows = pd.concat(raw.values(), ignore_index=True)
for col in ["sentiment_label", "sentiment_id", "sarcasm_label", "sarcasm_id",
            "script_type", "has_context", "split"]:
    print(f"--- {col} ---")
    print(allrows[col].value_counts(dropna=False).to_string(), "\n")

print("label text <-> id must be one-to-one:")
display(pd.crosstab(allrows.sentiment_label, allrows.sentiment_id))
display(pd.crosstab(allrows.sarcasm_label, allrows.sarcasm_id))

--- sentiment_label ---
sentiment_label
Negative    25749
Neutral     12782
Positive     7864 

--- sentiment_id ---
sentiment_id
0    25749
1    12782
2     7864 

--- sarcasm_label ---
sarcasm_label
0    31338
1    15057 

--- sarcasm_id ---
sarcasm_id
0    31338
1    15057 

--- script_type ---
script_type
sinhala_only    26534
latin_only      12924
mixed            6937 

--- has_context ---
has_context
True    46395 

--- split ---
split
train    32582
val       6938
test      6875 

label text <-> id must be one-to-one:


sentiment_id,0,1,2
sentiment_label,,,
Negative,25749,0,0
Neutral,0,12782,0
Positive,0,0,7864


sarcasm_id,0,1
sarcasm_label,,
0,31338,0
1,0,15057


In [4]:
# text_a / text_b are the encoder's input pair.
tr = raw["train"]
for a, b in [("text", "text_a"), ("context", "text_b"), ("context_input", "text_b"), ("context", "context_input")]:
    print(f"{a:13s} == {b:13s}: {(tr[a] == tr[b]).mean():7.2%} of train rows identical")

r = raw["test"].iloc[0]
for c in ["text", "text_a", "context", "context_input", "text_b"]:
    print(f"\n[{c}]\n{str(r[c])[:300]}")

text          == text_a       : 100.00% of train rows identical
context       == text_b       : 100.00% of train rows identical
context_input == text_b       : 100.00% of train rows identical
context       == context_input: 100.00% of train rows identical

[text]
දැන්නම් මෙයාව දැක්කත් හිනා වෙනවා බහුතරයක් මිනිස්සු,,,, (වහල්ලු හැර..

[text_a]
දැන්නම් මෙයාව දැක්කත් හිනා වෙනවා බහුතරයක් මිනිස්සු,,,, (වහල්ලු හැර..

[context]
ලංකාවේ පළමු වතාවට අනාගතය ගැන හිතන ආණ්ඩුවක්! අද අපි ණයක් ගන්නේ 2040දී අපිට ගෙවන්න පුළුවන්ද බලලා..- ජනපති #LKA #News1st #SL #AnuraKumaraDissanayake #KHM

[context_input]
ලංකාවේ පළමු වතාවට අනාගතය ගැන හිතන ආණ්ඩුවක්! අද අපි ණයක් ගන්නේ 2040දී අපිට ගෙවන්න පුළුවන්ද බලලා..- ජනපති #LKA #News1st #SL #AnuraKumaraDissanayake #KHM

[text_b]
ලංකාවේ පළමු වතාවට අනාගතය ගැන හිතන ආණ්ඩුවක්! අද අපි ණයක් ගන්නේ 2040දී අපිට ගෙවන්න පුළුවන්ද බලලා..- ජනපති #LKA #News1st #SL #AnuraKumaraDissanayake #KHM


In [5]:
# The bootstrap resamples parent posts. Check what post_group adds.
for s in SPLITS:
    df = raw[s]
    print(f"{s:5s} posts: {df.post_id.nunique():5d}   post_groups: {df.post_group.nunique():5d}   "
          f"comments/post (median): {df.groupby('post_id').size().median():.0f}")
pg = raw["test"].groupby("post_group").post_id.nunique()
print("\ntest: posts per post_group ->", pg.value_counts().sort_index().to_dict())

train posts:   476   post_groups:   472   comments/post (median): 78
val   posts:   101   post_groups:    98   comments/post (median): 79
test  posts:    99   post_groups:    97   comments/post (median): 79

test: posts per post_group -> {1: 95, 2: 2}


## Decisions for this conversion
Set these from the outputs above. **Every value here is a recorded design decision.**
- `CONTEXT_COL`: the caption text the encoder actually saw. The LLM must get the same text,
  otherwise encoder-vs-LLM is not a fair comparison. `"auto"` picks the column identical to `text_b`.
- Label/script maps: add any spelling Cell 4 printed. Unmapped values stop the notebook.

In [8]:
CONTEXT_COL = "auto"     # or "context" / "context_input" / "text_b"

if CONTEXT_COL == "auto":
    match = {c: (raw["train"][c] == raw["train"]["text_b"]).mean() for c in ("context", "context_input")}
    best = max(match, key=match.get)
    CONTEXT_COL = best if match[best] >= 0.99 else "text_b"
    print("identity with text_b:", {k: f"{v:.2%}" for k, v in match.items()})
print("CONTEXT_COL =", CONTEXT_COL)

SENT_MAP = {"negative": "negative", "neutral": "neutral", "positive": "positive",
            "neg": "negative", "neu": "neutral", "pos": "positive"}
SARC_MAP = {"sarcastic": 1, "non-sarcastic": 0, "non_sarcastic": 0, "nonsarcastic": 0,
            "not sarcastic": 0, "not_sarcastic": 0, "1": 1, "0": 0, "yes": 1, "no": 0}
SCRIPT_MAP = {"sinhala_only": "sinhala", "latin_only": "latin", "mixed": "mixed",
              "sinhala": "sinhala", "singlish": "latin", "latin": "latin", "romanized": "latin",
              "romanised": "latin", "code-mixed": "mixed", "code_mixed": "mixed"}

identity with text_b: {'context': '100.00%', 'context_input': '100.00%'}
CONTEXT_COL = context


In [9]:
def map_strict(series, mapping, name):
    key = series.astype(str).str.strip().str.lower()
    out = key.map(mapping)
    bad = sorted(key[out.isna()].unique())
    if bad:
        raise ValueError(f"unmapped {name} values: {bad[:10]} -> add them to the map in Cell 8")
    return out

def nfc(s):
    return unicodedata.normalize("NFC", s or "")

def convert(split, df):
    out = pd.DataFrame({
        "id":           df["id"].str.strip(),
        "post_id":      df["post_id"].str.strip(),
        "post_group":   df["post_group"].str.strip(),
        "text":         df["text"].map(nfc),
        "context_text": df[CONTEXT_COL].map(nfc),
        "script_type":  map_strict(df["script_type"], SCRIPT_MAP, "script_type"),
        "sentiment":    map_strict(df["sentiment_label"], SENT_MAP, "sentiment"),
        "is_sarcastic": map_strict(df["sarcasm_label"], SARC_MAP, "sarcasm").astype(int),
        "split":        split,
    })
    # the parquet's own split column must agree with the file it came from
    assert (df["split"].str.strip().str.lower().isin([split, {"val": "validation"}.get(split, split)])).all(), \
        f"{split}: rows whose 'split' column says otherwise"
    return out.reset_index(drop=True)

data = {s: convert(s, raw[s]) for s in SPLITS}
data["test"].head()

,id,post_id,post_group,text,context_text,script_type,sentiment,is_sarcastic,split
0,P20C1,P20,P20,"දැන්නම් මෙයාව දැක්කත් හිනා වෙනවා බහුතරයක් මිනිස්සු,,,, (වහල්ලු හැර..",ලංකාවේ පළමු වතාවට අනාගතය ගැන හිතන ආණ්ඩුවක්! අද අපි ණයක් ගන්නේ 2040දී අපිට ගෙ...,sinhala,negative,0,test
1,P20C2,P20,P20,අනෙ අපොයි🤣,ලංකාවේ පළමු වතාවට අනාගතය ගැන හිතන ආණ්ඩුවක්! අද අපි ණයක් ගන්නේ 2040දී අපිට ගෙ...,sinhala,negative,0,test
2,P20C3,P20,P20,ගොං හරකා,ලංකාවේ පළමු වතාවට අනාගතය ගැන හිතන ආණ්ඩුවක්! අද අපි ණයක් ගන්නේ 2040දී අපිට ගෙ...,sinhala,negative,0,test
3,P20C4,P20,P20,කථා සාප්පුවක් !,ලංකාවේ පළමු වතාවට අනාගතය ගැන හිතන ආණ්ඩුවක්! අද අපි ණයක් ගන්නේ 2040දී අපිට ගෙ...,sinhala,negative,1,test
4,P20C5,P20,P20,ඕයී ඕයී 🤣🤣🤣🤭,ලංකාවේ පළමු වතාවට අනාගතය ගැන හිතන ආණ්ඩුවක්! අද අපි ණයක් ගන්නේ 2040දී අපිට ගෙ...,sinhala,neutral,0,test


In [10]:
for s in SPLITS:
    sent_pairs = pd.crosstab(data[s].sentiment, raw[s].sentiment_id.values)
    sarc_pairs = pd.crosstab(data[s].is_sarcastic, raw[s].sarcasm_id.values)
    one_to_one = lambda t: ((t > 0).sum(axis=0) == 1).all() and ((t > 0).sum(axis=1) == 1).all()
    assert one_to_one(sent_pairs) and one_to_one(sarc_pairs), f"{s}: label text and id disagree"

SENT_ID_ORDER = pd.crosstab(data["train"].sentiment, raw["train"].sentiment_id.values).idxmax(axis=0).to_dict()
print("Encoder sentiment id -> label:", SENT_ID_ORDER, " <- needed to read E1 prediction files")

Encoder sentiment id -> label: {'0': 'negative', '1': 'neutral', '2': 'positive'}  <- needed to read E1 prediction files


In [11]:
for s in SPLITS:
    listed, present = set(ids_raw[s]["id"].str.strip()), set(data[s].id)
    print(f"{s:5s} id-file {len(listed):6,}  parquet {len(present):6,}  "
          f"missing {len(listed - present):4}  extra {len(present - listed):4}")
    assert listed == present, f"{s}: parquet rows do not match {ID_FILES[s].name}"
print("ID files and parquet splits agree.")

train id-file 32,582  parquet 32,582  missing    0  extra    0
val   id-file  6,938  parquet  6,938  missing    0  extra    0
test  id-file  6,875  parquet  6,875  missing    0  extra    0
ID files and parquet splits agree.


In [12]:
problems = []
for s, df in data.items():
    for col in ("id", "post_id", "text"):
        n = (df[col].str.strip() == "").sum()
        if n:
            problems.append(f"{s}: {n} empty {col}")
    if df.id.duplicated().any():
        problems.append(f"{s}: {df.id.duplicated().sum()} duplicate ids")
    n_ctx = (df.context_text.str.strip() == "").sum()
    if n_ctx:
        problems.append(f"NOTE {s}: {n_ctx} rows without context ({n_ctx/len(df):.1%})")
print("\n".join(problems) or "No record-level problems.")

No record-level problems.


In [13]:
EXPECTED = {
    "train size": 32_582, "val size": 6_938, "test size": 6_875,
    "test negative": 3_763, "test neutral": 1_908, "test positive": 1_204,
    "test sarcastic": 2_277,
    "test sarcastic negative": 1_815, "test sarcastic neutral": 344, "test sarcastic positive": 118,
    "test posts": 99,
}
te = data["test"]; sar = te[te.is_sarcastic == 1]
LABELS = ("negative", "neutral", "positive")
got = {
    "train size": len(data["train"]), "val size": len(data["val"]), "test size": len(te),
    **{f"test {l}": int((te.sentiment == l).sum()) for l in LABELS},
    "test sarcastic": len(sar),
    **{f"test sarcastic {l}": int((sar.sentiment == l).sum()) for l in LABELS},
    "test posts": te.post_id.nunique(),
}
checks = pd.DataFrame({"expected": EXPECTED, "got": got})
checks["ok"] = checks.expected == checks.got
display(checks)
assert checks.ok.all(), "§5.1 counts do not match — STOP and fix before anything else"
print("§5.1 count checks: ALL PASS")

,expected,got,ok
train size,32582,32582,True
val size,6938,6938,True
test size,6875,6875,True
test negative,3763,3763,True
test neutral,1908,1908,True
test positive,1204,1204,True
test sarcastic,2277,2277,True
test sarcastic negative,1815,1815,True
test sarcastic neutral,344,344,True
test sarcastic positive,118,118,True


§5.1 count checks: ALL PASS


In [14]:
for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    shared = {k: len(set(data[a][k]) & set(data[b][k])) for k in ("id", "post_id", "post_group")}
    print(f"{a:5s}–{b:5s}  shared ids: {shared['id']:5d}  post_ids: {shared['post_id']:5d}  "
          f"post_groups: {shared['post_group']:5d}")
assert not (set(data["train"].id) & set(data["test"].id)), "train/test share ids"
# Shared post_ids/post_groups are not an error if the thesis split by comment,
# but retrieval (notebook 05) must then also exclude neighbours from the query's post.

train–val    shared ids:     0  post_ids:     0  post_groups:     0
train–test   shared ids:     0  post_ids:     0  post_groups:     0
val  –test   shared ids:     0  post_ids:     0  post_groups:     0


In [15]:
both = pd.concat(data.values(), ignore_index=True)
desc = (both.assign(sarcasm=both.is_sarcastic.map({0: "non-sarc", 1: "sarc"}))
            .pivot_table(index=["split", "sarcasm"], columns="sentiment", values="id",
                         aggfunc="count", fill_value=0)
            .reindex(columns=list(LABELS)))
desc["total"] = desc.sum(axis=1)
display(desc)
display(pd.crosstab(te.script_type, [te.is_sarcastic, te.sentiment], margins=True))
print("Sarcasm base rate (train):", round(data["train"].is_sarcastic.mean(), 4), " (expect ≈ 0.3275)")

sentiment       negative  neutral  positive  total
split sarcasm                                     
test  non-sarc      1948     1564      1086   4598
      sarc          1815      344       118   2277
train non-sarc     10279     6914      4873  22066
      sarc          7986     2057       473  10516
val   non-sarc      2025     1485      1164   4674
      sarc          1696      418       150   2264

is_sarcastic        0                         1                    All
sentiment    negative neutral positive negative neutral positive      
script_type                                                           
latin             490     634      311      335      83       24  1877
mixed             271     303      174      251      71       17  1087
sinhala          1187     627      601     1229     190       77  3911
All              1948    1564     1086     1815     344      118  6875

Sarcasm base rate (train): 0.3228  (expect ≈ 0.3275)


In [16]:
FIELDS = ["id", "post_id", "post_group", "text", "context_text", "script_type",
          "sentiment", "is_sarcastic", "split"]
for s, df in data.items():
    path = PROC_DIR / f"{s}.jsonl"
    with open(path, "w", encoding="utf-8") as f:
        for rec in df[FIELDS].to_dict("records"):
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"wrote {path.relative_to(ROOT)}  ({len(df):,} rows)")

back = pd.read_json(PROC_DIR / "test.jsonl", lines=True, dtype=False)
assert len(back) == len(te) and back.text.tolist() == te.text.tolist() and back.id.astype(str).tolist() == te.id.tolist()
print("Round-trip OK (Sinhala text unchanged)")

wrote data\processed\train.jsonl  (32,582 rows)
wrote data\processed\val.jsonl  (6,938 rows)
wrote data\processed\test.jsonl  (6,875 rows)
Round-trip OK (Sinhala text unchanged)


In [17]:
rng = np.random.default_rng(SEED)

def stratified_take(df, n, by):
    """n rows, proportional across `by` (largest-remainder allocation)."""
    if n >= len(df):
        return df
    share = df[by].value_counts(normalize=True).sort_index() * n
    alloc = np.floor(share).astype(int)
    for k in (share - alloc).sort_values(ascending=False).index[: n - alloc.sum()]:
        alloc[k] += 1
    return pd.concat([df[df[by] == k].sample(n=int(m), random_state=int(rng.integers(2**31)))
                      for k, m in alloc.items() if m > 0])

# Option B: every sarcastic test row + the same number of non-sarcastic rows, stratified by script
optB = pd.concat([sar, stratified_take(te[te.is_sarcastic == 0], len(sar), "script_type")])

# Pilot: 300 validation rows. [DECISION] sarcastic Neutral/Positive over-represented (H5).
PILOT_ALLOC = {(1, "negative"): 80, (1, "neutral"): 60, (1, "positive"): 40,
               (0, "negative"): 40, (0, "neutral"): 40, (0, "positive"): 40}
va = data["val"]
parts = []
for (is_s, sent), n in PILOT_ALLOC.items():
    cell = va[(va.is_sarcastic == is_s) & (va.sentiment == sent)]
    assert len(cell) >= n, f"val cell {(is_s, sent)} has only {len(cell)} rows"
    parts.append(stratified_take(cell, n, "script_type"))
pilot = pd.concat(parts)

def write_ids(ids, name):
    ids = sorted(ids)
    (EVAL_DIR / name).write_text("\n".join(ids) + "\n", encoding="utf-8")
    print(f"wrote data/eval_sets/{name}  ({len(ids):,} ids)")

write_ids(te.id, "test_full.txt")
write_ids(optB.id, "test_optB.txt")
write_ids(pilot.id, "pilot_val_300.txt")

wrote data/eval_sets/test_full.txt  (6,875 ids)
wrote data/eval_sets/test_optB.txt  (4,554 ids)
wrote data/eval_sets/pilot_val_300.txt  (300 ids)


In [18]:
assert optB.id.is_unique and pilot.id.is_unique and len(pilot) == 300
assert (optB.is_sarcastic == 1).sum() == (optB.is_sarcastic == 0).sum() == 2277
display(pd.crosstab(optB.script_type, optB.is_sarcastic, margins=True))
display(pd.crosstab([pilot.is_sarcastic, pilot.sentiment], pilot.script_type, margins=True))

is_sarcastic,0,1,All
script_type,,,
latin,711,442,1153
mixed,370,339,709
sinhala,1196,1496,2692
All,2277,2277,4554


script_type             latin  mixed  sinhala  All
is_sarcastic sentiment                            
0            negative      10      6       24   40
             neutral       17      6       17   40
             positive      12      6       22   40
1            negative      16     10       54   80
             neutral       14      9       37   60
             positive       9      6       25   40
All                        78     43      179  300

In [19]:
print(f"""## {datetime.date.today()} · Data preparation (notebook 01)
- Source: {', '.join(p.name for p in RAW_FILES.values())} + *_ids.csv (ID sets match exactly)
- §5.1 counts: all pass. Test posts: {te.post_id.nunique()}; test post_groups: {te.post_group.nunique()}.
- Context column given to the LLM: `{CONTEXT_COL}` (same text the encoder saw).
- Script type: taken from the thesis parquet (`script_type`), mapped {sorted(set(SCRIPT_MAP.values()))}.
- Encoder sentiment ids: {SENT_ID_ORDER}
- Eval sets (seed {SEED}): test_full {len(te)}, option B {len(optB)}, pilot {len(pilot)} {PILOT_ALLOC}
- Env: Python {sys.version.split()[0]}, pandas {pd.__version__}, numpy {np.__version__}, {platform.system()}
""")

## 2026-11-21 · Data preparation (notebook 01)
- Source: train_v2.0.parquet, val_v2.0.parquet, test_v2.0.parquet + *_ids.csv (ID sets match exactly)
- §5.1 counts: all pass. Test posts: 99; test post_groups: 97.
- Context column given to the LLM: `context` (same text the encoder saw).
- Script type: taken from the thesis parquet (`script_type`), mapped ['latin', 'mixed', 'sinhala'].
- Encoder sentiment ids: {'0': 'negative', '1': 'neutral', '2': 'positive'}
- Eval sets (seed 42): test_full 6875, option B 4554, pilot 300 {(1, 'negative'): 80, (1, 'neutral'): 60, (1, 'positive'): 40, (0, 'negative'): 40, (0, 'neutral'): 40, (0, 'positive'): 40}
- Env: Python 3.13.15, pandas 3.1.1, numpy 2.5.5, Windows

